<a href="https://colab.research.google.com/github/ErrPeppe/OhnomiTools/blob/main/notebooks/01_fase1_data_ingestion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fase 1 — Acquisizione Dati e Pipeline ETL (`Bronze` $\to$ `Silver`)

Questo notebook documenta la preparazione del Data Lake per l'analisi evolutiva e topologica degli **Ohno-lncRNA** umani tramite la libreria **`OhnomiTools`**.

### Architettura della Pipeline
* **Codice (`GitHub`):** libreria installabile `OhnomiTools` (`Polars` + `Apache Parquet` compresso `zstd`).
* **Storage (`Google Drive`):** architettura *Medallion* in `UniTo/Tesi/Data/`:
  * `bronze/`: database sorgente grezzi (LncBook v2.0, dump OHNOLOGS, TSV scaricati da Ensembl BioMart).
  * `silver/`: tabelle pulite, normalizzate su cromosomi canonici (`chr1`–`chr22`, `chrX`, `chrY`) e salvate in formato `.parquet`.
  * `gold/`: tabelle statistiche finali e grafici pronti per il report di tesi.

### Database integrati
1. **LncBook v2.0 (GRCh38):** catalogo curato dei 95.243 geni lncRNA, filostratigrafia evolutiva (`Age` tramite sintenia nei vertebrati), interazioni lncRNA–miRNA ed espressione tissutale.
2. **OHNOLOGS (`homo_sapiens`):** lista degli ohnologhi umani derivati da duplicazioni dell'intero genoma (*1R/2R-WGD*) e coppie intra-umane mappate da `ENSP` a `ENSG`.
3. **Ensembl BioMart (Archivi HTTPS):** catalogo completo dei geni `protein_coding` umani (con `strand` `+`/`-`) e gruppo di controllo **SSD** (*Small Scale Duplications*), ottenuto sottraendo via `anti-join` tutte le coppie WGD dall'insieme dei paraloghi umani.

## 1. Inizializzazione dell'Ambiente Cloud (`OhnomiTools` + Google Drive)
Installazione della libreria da GitHub e montaggio automatico della cartella dati su Google Drive tramite `setup_colab()`, con stampa dello stato di presenza e dimensione di tutti i file `bronze` e `silver`.

In [ ]:
!pip install -q --no-cache-dir git+https://github.com/ErrPeppe/OhnomiTools.git
from ohnomitools import setup_colab
PATHS = setup_colab()

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
shape: (20, 5)
┌────────┬─────────────────────────────────┬──────────┬─────────┬──────────────────────────────────┐
│ Layer  ┆ Dataset                         ┆ Presente ┆ Size_MB ┆ Path                             │
│ ---    ┆ ---                             ┆ ---      ┆ ---     ┆ ---                              │
│ str    ┆ str                             ┆ str      ┆ f64     ┆ str                              │
╞════════╪═════════════════════════════════╪══════════╪═════════╪══════════════════════════════════╡
│ Bronze ┆ LncBook v2.0 GTF                ┆ SI       ┆ 15.2    ┆ bronze/lncRNA_LncBookv2.0_GRCh38 │
│        ┆                                 ┆          ┆         ┆ .gtf.gz                          │
│ Bronze ┆ LncBook v2.0 Conservation       ┆ SI       ┆ 46.78   ┆ bronze/conservation_LncBook2.0.c │
│        ┆                                

## 2. Esecuzione della Pipeline ETL (`Bronze` $\to$ `Silver`)
Conversione del GTF di LncBook v2.0 (estrazione degli attributi dalla colonna 9), streaming out-of-core dei CSV di conservazione/interazioni/espressione, normalizzazione dei file OHNOLOGS privi di intestazione e download filtrato (`protein_coding`, cromosomi canonici) dei paraloghi umani da Ensembl BioMart.

In [ ]:
from ohnomitools.fetcher import DataIngestionPipeline

pipeline = DataIngestionPipeline(paths=PATHS)
pipeline.fetch_and_build_ensembl_pc_genes(force_rebuild=False)
pipeline.build_ohnologs_wgd_tables(force_rebuild=True)
silver_files = pipeline.run_all(force_rebuild=False)
PATHS.print_status()

shape: (20, 5)
┌────────┬─────────────────────────────────┬──────────┬─────────┬──────────────────────────────────┐
│ Layer  ┆ Dataset                         ┆ Presente ┆ Size_MB ┆ Path                             │
│ ---    ┆ ---                             ┆ ---      ┆ ---     ┆ ---                              │
│ str    ┆ str                             ┆ str      ┆ f64     ┆ str                              │
╞════════╪═════════════════════════════════╪══════════╪═════════╪══════════════════════════════════╡
│ Bronze ┆ LncBook v2.0 GTF                ┆ SI       ┆ 15.2    ┆ bronze/lncRNA_LncBookv2.0_GRCh38 │
│        ┆                                 ┆          ┆         ┆ .gtf.gz                          │
│ Bronze ┆ LncBook v2.0 Conservation       ┆ SI       ┆ 46.78   ┆ bronze/conservation_LncBook2.0.c │
│        ┆                                 ┆          ┆         ┆ sv.gz                            │
│ Bronze ┆ LncBook v2.0 miRNA Interactions ┆ SI       ┆ 1.6     ┆ bronze/lnc

## 3. Controllo Qualità (QC) dei Dataset WGD e SSD
Verifica dei volumi finali estratti in `silver/` e ispezione dello schema delle coppie **WGD** (OHNOLOGS) e **SSD pure** (Ensembl BioMart dopo sottrazione delle WGD), incluse coordinate genomiche, `strand` e clade dell'ultimo antenato comune (LCA).

In [ ]:
import polars as pl

df_wgd_g = pl.read_parquet(PATHS.silver_wgd_genes)
df_wgd_p = pl.read_parquet(PATHS.silver_wgd_pairs)
df_ssd_p = pl.read_parquet(PATHS.silver_ssd_pairs)

print(f"Geni WGD umani (OHNOLOGS)    : {len(df_wgd_g):,}")
print(f"Coppie WGD intra-umane       : {len(df_wgd_p):,}")
print(f"Coppie SSD pure (Ensembl)    : {len(df_ssd_p):,}")
display(df_wgd_p.head(3))
display(df_ssd_p.head(3))

Geni WGD umani (OHNOLOGS)    : 4,602
Coppie WGD intra-umane       : 4,221
Coppie SSD pure (Ensembl)    : 127,578


protein_id_1,protein_id_2,wgd_round,gene_id_1,gene_name_1,chr_1,start_1,end_1,strand_1,family_id_1,retention_category_1,gene_id_2,gene_name_2,chr_2,start_2,end_2,strand_2,family_id_2,retention_category_2,pair_key
str,str,str,str,str,str,i64,i64,str,str,str,str,str,str,i64,i64,str,str,str,str
"""ENSP00000367830""","""ENSP00000295797""","""unk""","""ENSG00000067606""","""PRKCZ""","""chr1""",2050411,2185395,"""+""","""OG0002838""","""unk""","""ENSG00000163558""","""PRKCI""","""chr3""",170222424,170305977,"""+""","""OG0002838""","""unk""","""ENSG00000067606_ENSG0000016355…"
"""ENSP00000367668""","""ENSP00000478173""","""unk""","""ENSG00000142606""","""MMEL1""","""chr1""",2590639,2633016,"""-""","""OG0000142""",null,"""ENSG00000196549""","""MME""","""chr3""",155024124,155183704,"""+""","""OG0000142""",null,"""ENSG00000142606_ENSG0000019654…"
"""ENSP00000270722""","""ENSP00000498411""","""unk""","""ENSG00000142611""","""PRDM16""","""chr1""",3069168,3438621,"""+""","""OG0002282""",null,"""ENSG00000085276""","""MECOM""","""chr3""",169083499,169663775,"""-""","""OG0002282""",null,"""ENSG00000085276_ENSG0000014261…"


gene_id_1,gene_id_2,Paralogue last common ancestor with Human,homology_type,perc_id_1,perc_id_2,pair_key,gene_name_1,chr_1,start_1,end_1,strand_1,gene_name_2,chr_2,start_2,end_2,strand_2
str,str,str,str,f64,f64,str,str,str,i64,i64,str,str,str,i64,i64,str
"""ENSG00000137177""","""ENSG00000196169""","""Opisthokonta""","""other_paralog""",11.8006,21.3427,"""ENSG00000137177_ENSG0000019616…","""KIF13A""","""chr6""",17759183,17987635,"""-""","""KIF19""","""chr17""",74326188,74355835,"""+"""
"""ENSG00000214940""","""ENSG00000234719""","""Homininae""","""within_species_paralog""",72.3577,67.2544,"""ENSG00000214940_ENSG0000023471…","""NPIPA8""","""chr16""",18317919,18336736,"""-""","""NPIPB2""","""chr16""",11927259,11976643,"""-"""
"""ENSG00000167858""","""ENSG00000092850""","""Bilateria""","""within_species_paralog""",32.5359,31.6279,"""ENSG00000092850_ENSG0000016785…","""TEKT1""","""chr17""",6789133,6831761,"""-""","""TEKT2""","""chr1""",36084094,36088278,"""+"""
